In [ ]:
pip install numpy==1.26.4

In [ ]:
pip install scikit-learn==1.4.2

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
import json


years_historical = np.array([2020, 2021, 2022, 2023, 2024, 2025])
consumption_historical = np.array([760000, 720000, 810000, 860000, 920000, 1000000])
students_historical = np.array([950, 900, 1000, 1050, 1100, 1200])


X_hist = years_historical.reshape(-1, 1)
y_hist = consumption_historical

model = LinearRegression()
model.fit(X_hist, y_hist)


future_years = np.array([2026, 2027, 2028, 2029, 2030, 2031, 2032])
X_future = future_years.reshape(-1, 1)
predicted_consumption = model.predict(X_future)

predicted_consumption = predicted_consumption.astype(int)


# 500 kW installed in 2028 (Phase 3 of roadmap)
solar_capacity_kw = 500
sun_hours_per_day = 5.5       # Indore average (Central India)
system_efficiency = 0.85      # 15% losses (heat, dust, inverter)
installation_year = 2028

def solar_generation(year):
    if year >= installation_year:
        return int(solar_capacity_kw * sun_hours_per_day * 365 * system_efficiency)
    return 0

solar_projected = [solar_generation(y) for y in future_years]

# 5. NET ENERGY (Consumption - Solar)
net_energy = [max(0, predicted_consumption[i] - solar_projected[i])
              for i in range(len(future_years))]

INDIA_GRID_EMISSION_FACTOR = 0.82  # kg CO₂ per kWh (CEA 2024)

def carbon_emissions(kwh):
    return round((kwh * INDIA_GRID_EMISSION_FACTOR) / 1000, 1)  # in tonnes

carbon_from_grid = [carbon_emissions(ne) for ne in net_energy]

TARIFF_PER_KWH = 8.0  # ₹8/kWh — MPERC 2025-26 commercial rate

def annual_cost(kwh):
    return int(kwh * TARIFF_PER_KWH)

annual_bill = [annual_cost(ne) for ne in net_energy]
baseline_bill = annual_cost(1000000)

# 8. NET ZERO DETECTION
net_zero_year = None
for i, y in enumerate(future_years):
    if solar_projected[i] >= predicted_consumption[i]:
        net_zero_year = int(y)
        break


# Flag years where consumption grows faster than expected (>7% YoY)
anomaly_flags = []
for i in range(1, len(predicted_consumption)):
    yoy_growth = (predicted_consumption[i] - predicted_consumption[i-1]) / predicted_consumption[i-1]
    anomaly_flags.append({
        "year": int(future_years[i]),
        "yoy_growth_pct": round(yoy_growth * 100, 2),
        "flag": " HIGH GROWTH" if yoy_growth > 0.07 else " Normal"
    })

#
forecast_report = {
    "model": "Linear Regression",
    "training_years": years_historical.tolist(),
    "training_consumption_kwh": consumption_historical.tolist(),
    "model_slope": round(model.coef_[0], 2),
    "model_intercept": round(model.intercept_, 2),
    "r_squared": round(model.score(X_hist, y_hist), 4),
    "baseline_2025": {
        "consumption_kwh": 1000000,
        "annual_bill_inr": baseline_bill,
        "carbon_emissions_tonnes": carbon_emissions(1000000)
    },
    "forecast": [],
    "net_zero_year": net_zero_year if net_zero_year else "Post-2032 (increase solar capacity)"
}

for i, year in enumerate(future_years):
    forecast_report["forecast"].append({
        "year": int(year),
        "predicted_consumption_kwh": int(predicted_consumption[i]),
        "solar_generation_kwh": solar_projected[i],
        "net_grid_dependency_kwh": net_energy[i],
        "carbon_emissions_tonnes": carbon_from_grid[i],
        "annual_electricity_bill_inr": annual_bill[i],
        "savings_vs_baseline_inr": baseline_bill - annual_bill[i],
        "renewable_offset_pct": round((solar_projected[i] / predicted_consumption[i]) * 100, 1) if solar_projected[i] > 0 else 0.0
    })

# 11. PRINT RESULTS
print("=" * 65)
print("   NMIMS INDORE — ENERGY DEMAND FORECAST (2026–2032)")
print("=" * 65)
print(f"Model: Linear Regression | R² Score: {forecast_report['r_squared']}")
print(f"Baseline 2025: 10,00,000 kWh | ₹{baseline_bill:,}/year | {carbon_emissions(1000000)} tonnes CO₂")
print(f"Solar Install: {solar_capacity_kw} kW rooftop | Target Year: {installation_year}")
print("=" * 65)

print(f"\n{'Year':<6} {'Demand(kWh)':<14} {'Solar(kWh)':<12} {'Net Grid':<12} {'CO₂(T)':<10} {'Bill(₹L)':<12} {'Offset%'}")
print("-" * 75)

for entry in forecast_report["forecast"]:
    bill_lakhs = round(entry["annual_electricity_bill_inr"] / 100000, 1)
    print(f"{entry['year']:<6} {entry['predicted_consumption_kwh']:<14,} "
          f"{entry['solar_generation_kwh']:<12,} "
          f"{entry['net_grid_dependency_kwh']:<12,} "
          f"{entry['carbon_emissions_tonnes']:<10} "
          f"₹{bill_lakhs:<10} "
          f"{entry['renewable_offset_pct']}%")

print("-" * 75)
print(f"\n NET-ZERO TARGET YEAR: {forecast_report['net_zero_year']}")
print(f"\n ANOMALY DETECTION:")
for a in anomaly_flags:
    print(f"   {a['year']}: YoY Growth = {a['yoy_growth_pct']}% → {a['flag']}")

print("\n Forecast engine complete. JSON report ready for dashboard API.")



   NMIMS INDORE — ENERGY DEMAND FORECAST (2026–2032)
Model: Linear Regression | R² Score: 0.9063
Baseline 2025: 10,00,000 kWh | ₹8,000,000/year | 820.0 tonnes CO₂
Solar Install: 500 kW rooftop | Target Year: 2028

Year   Demand(kWh)    Solar(kWh)   Net Grid     CO₂(T)     Bill(₹L)     Offset%
---------------------------------------------------------------------------
2026   1,030,000      0            1,030,000    844.6      ₹82.4       0.0%
2027   1,082,857      0            1,082,857    887.9      ₹86.6       0.0%
2028   1,135,714      853,187      282,527      231.7      ₹22.6       75.1%
2029   1,188,571      853,187      335,384      275.0      ₹26.8       71.8%
2030   1,241,428      853,187      388,241      318.4      ₹31.1       68.7%
2031   1,294,285      853,187      441,098      361.7      ₹35.3       65.9%
2032   1,347,142      853,187      493,955      405.0      ₹39.5       63.3%
---------------------------------------------------------------------------

 NET-ZERO TARGET